In [1]:
version = "REPLACE_PACKAGE_VERSION"

# Assignment 2: Mining Itemsets (Part II)

## Finding Frequent Itemsets with Apriori

In Part I of this assignment, the summary statistics gave us a brief view of what the data look like. Now it is time for the real business - let's use the *Apriori* algorithm to find the frequent itemsets. 
First, let's import the packages and dependencies that will be used in this part of the assignment.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

from mlxtend.frequent_patterns import apriori

from mads.lib.path import assets

In [3]:
# Ignore warnings generated by mlxtend

**<span style="color:red">NOTE: These are all the imports we need to make for this assignment. You should not make other imports in your submitted notebook. You will receive 0 points for the exercises if your solution includes additional imports.</span>**

Before you practice Apriori on the Instacart dataset, let's use another dataset as an example to get familiar with the algorithm. For this assignment, **we sampled 10,000 Tweets with two or more food/drink emojis** (yes, those colorful tasty ideograms). You will represent this dataset as a collection of itemsets and practice what we learned in class.

In [4]:
# Load the tweets data from files.
file_tweets = assets.find("food_drink_emoji_tweets.txt")
tweets_df = pd.read_csv(file_tweets, sep="\t",header=None)
tweets_df.columns = ['text']

# Only the emojis below are considered in this assignment.
emoji_list = "🍇🍈🍉🍊🍋🍌🍍🥭🍎🍏🍐🍑🍒🍓🥝🍅🥥🥑🍆🥔🥕🌽🌶🥒🥬🥦🍄🥜🌰🍞🥐🥖🥨🥯🥞🧀🍖🍗🥩🥓🍔🍟🍕🌭🥪🌮🌯🥙🥚🍳🥘🍲🥣🥗🍿🧂🥫🍱🍘🍙🍚🍛🍜🍝🍠🍢🍣🍤🍥🥮🍡🥟🥠🥡🦀🦞🦐🦑🍦🍧🍨🍩🍪🎂🍰🧁🥧🍫🍬🍭🍮🍯🍼🥛☕🍵🍶🍾🍷🍸🍹🍺🍻🥂🥃"
emoji_set = set(emoji_list)

# Get a list of emojis used in each tweet.
tweets_df['emojis'] = tweets_df.text.apply(lambda text:np.unique([chr for chr in text if chr in emoji_set]))

# Create a matrix for Apriori algorithms.
mlb = MultiLabelBinarizer()
emoji_data = mlb.fit_transform(tweets_df.emojis).astype(bool)
emoji_matrix = pd.DataFrame(data=emoji_data, index=tweets_df.index, columns=mlb.classes_)

We can call the Apriori API now and specify the minimal support we want. You may learn more about this API from its [documentation](http://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/).

In [5]:
# Apply the Apriori algorithms and find the most frequent items.
freq_emoji_itemsets = apriori(emoji_matrix, min_support=0.005, use_colnames=True)
freq_emoji_itemsets.sort_values("support", ascending=False).head()

,support,itemsets
55,0.181663,frozenset({🍻})
59,0.148725,frozenset({🎂})
22,0.138783,frozenset({🍔})
57,0.108255,frozenset({🍾})
60,0.103234,frozenset({🥂})


With the above command, we preserve all itemsets with a min support of 0.005 (half percent of the shopping baskets). We can now use the following command to extract the frequent itemsets with length 2 and beyond.

In [6]:
# Apply the Apriori algorithms and find the most frequent 2-itemsets.
freq_emoji_itemsets[freq_emoji_itemsets["itemsets"].apply(
    lambda x: len(x) > 1)].sort_values("support", ascending=False).head()


,support,itemsets
136,0.081944,"frozenset({🍻, 🍔})"
202,0.057040,"frozenset({🍾, 🥂})"
169,0.045993,"frozenset({🎂, 🍰})"
192,0.042478,"frozenset({🍻, 🍺})"
92,0.036654,"frozenset({🍆, 🍑})"


**Now, it is time to apply the Apriori algorithm to the Instacart dataset.**

Since we have already shown you how to transform the Instacart dataset into itemsets, we concatenate the data preprocessing code into one block. Please run the following code block to load and preprocess the data. Notice that we limit the number of products in this problem to 100 by popularity so that the apriori algorithm will not run for a long time.

In [7]:
# Load data from files.
file_orders = assets.find("orders.csv.zip")
file_products = assets.find("products.csv.zip")
orders = pd.read_csv(file_orders, nrows=10000)
products = pd.read_csv(file_products)

# Group orders by order id and merge them into a list.
order_baskets = orders.groupby("order_id")["product_id"].apply(list)

# Convert the above pandas Series to a pandas DataFrame.
order_baskets = order_baskets.to_frame(name="products_id")

# Create the name map for later reference.
product_name_map = dict(zip(products["product_id"], products["product_name"]))

# Create the matrix like Part 1.
mlb = MultiLabelBinarizer()
data = mlb.fit_transform(order_baskets["products_id"]).astype(bool)
prod_matrix = pd.DataFrame(
    data=data, index=order_baskets.index, columns=mlb.classes_
)
prod_popularity = prod_matrix.sum(axis=0)
prod_matrix = prod_matrix.astype(bool)
top_prods = prod_popularity.sort_values(ascending=False).head(100).index
prod_matrix = prod_matrix[top_prods]
prod_matrix.head()

,24852,13176,21137,21903,47209,47766,16797,47626,22935,27966,...,27521,16185,41220,5025,5450,23375,20114,31915,21709,33198
order_id,,,,,,,,,,,,,,,,,,,,,
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5,False,True,False,False,True,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [8]:
prod_matrix.shape

(977, 100)

### Exercise 2.  (20 pts)
Complete the following `prod_frequent_itemsets` function to find all the **frequent *k*-itemsets** with a minimal support of **`min_support`** in the Instacart dataset. 

Your function should return a Pandas DataFrame object similar to the `emoji_frequent_itemsets` object above. The DataFrame object should have two columns: 
- The first one is named `support` and stores the support of the frequent itemsets. 
- The second column is named `itemsets` and stores the frequent itemset as a *frozenset* (the default return type of the apriori API).

Make sure that you are only returning the frequent itemsets that have the specified number of products (`k`).

In [9]:
def prod_frequent_itemsets(prod_matrix, min_support=0.005, k=3):
    # mine frequent itemsets w/ product IDs kept as column names
    frequent_itemsets = apriori(prod_matrix, min_support = min_support, use_colnames = True)
    # keep only the itemsets that contain exactly k products
    frequent_itemsets = frequent_itemsets[
        frequent_itemsets["itemsets"].apply(lambda x:len(x) == k)
    ]

    return frequent_itemsets

What it does:

* Runs `apriori(...)` on the Instacart product matrix
* Uses `use_colnames=True` so itemsets contain product IDs instead of column positions
* Filters results to only frequent itemsets of size `k`

This matches the assignment requirement that the returned DataFrame contains:

* `support`
* `itemsets`

and only includes itemsets with exactly `k` products.


If you implemented this function correctly, we can obtain all frequent 3-itemsets with a min support of 0.004 by running the following command.

In [10]:
prod_frequent_3itemsets = prod_frequent_itemsets(prod_matrix, min_support=0.004, k=3)
prod_frequent_3itemsets

,support,itemsets
372,0.004094,"frozenset({21137, 24852, 21903})"
373,0.004094,"frozenset({24852, 47766, 21903})"
374,0.005118,"frozenset({45007, 24852, 21903})"
375,0.004094,"frozenset({19057, 24852, 21903})"
376,0.005118,"frozenset({25890, 24852, 21903})"
377,0.004094,"frozenset({45066, 24852, 16797})"
378,0.004094,"frozenset({45066, 24852, 27845})"
379,0.007165,"frozenset({13176, 21137, 47209})"
380,0.004094,"frozenset({13176, 47209, 27966})"
381,0.004094,"frozenset({43352, 12341, 16797})"


In [11]:
prod_frequent_3itemsets = prod_frequent_itemsets(prod_matrix,
                                                 min_support=0.004,
                                                 k=3)
for row in prod_frequent_3itemsets.itertuples():
    assert row.support >= 0.004, f"[Exercise 2] The support of the itemset {row.itemsets} is below the threshold."
    assert len(row.itemsets) == 3, f"[Exercise 2] The itemset {row.itemsets} is not a 3-itemset."


Let's replace the product IDs with their names. Does the result make sense to you?

In [12]:
prod_frequent_3itemsets.itemsets = prod_frequent_3itemsets.itemsets.apply(lambda row: [product_name_map[i] for i in row])
prod_frequent_3itemsets

,support,itemsets
372,0.004094,"[Organic Strawberries, Banana, Organic Baby Sp..."
373,0.004094,"[Banana, Organic Avocado, Organic Baby Spinach]"
374,0.005118,"[Organic Zucchini, Banana, Organic Baby Spinach]"
375,0.004094,"[Organic Large Extra Fancy Fuji Apple, Banana,..."
376,0.005118,"[Boneless Skinless Chicken Breasts, Banana, Or..."
377,0.004094,"[Honeycrisp Apple, Banana, Strawberries]"
378,0.004094,"[Honeycrisp Apple, Banana, Organic Whole Milk]"
379,0.007165,"[Bag of Organic Bananas, Organic Strawberries,..."
380,0.004094,"[Bag of Organic Bananas, Organic Hass Avocado,..."
381,0.004094,"[Raspberries, Hass Avocados, Strawberries]"
